# Step 5: SHAP Top 3

# Mounting Google Drive

In [1]:
from google.colab import drive
drive.mount('/gdrive')
%cd /gdrive

Mounted at /gdrive
/gdrive


# Move to the Dataset Directory in My Drive

In [2]:
import os
os.chdir("/gdrive/My Drive/Machine Learning (HPL) - MITACS 2025")
!pwd

/gdrive/My Drive/Machine Learning (HPL) - MITACS 2025


# SHAP Top 3 Code:

In [3]:
"""
Created on Fri Jul 12 08:41:03 2024

@author: roqui
"""

import pandas as pd              # for reading the .csv file and related operations
import numpy as np               # for working with arrays (multi-dimensional)
import seaborn as sns
import matplotlib.pyplot as plt
import os

def plot_shap_ranking(df, title, filename):

    # Calculate mean SHAP values for each feature to determine overall importance
    feature_shap_values = pd.concat([
        df[['Feature_Top_1', 'Value_Top_1', 'Group']].rename(columns={'Feature_Top_1': 'Feature', 'Value_Top_1': 'SHAP_Value'}),
        df[['Feature_Top_2', 'Value_Top_2', 'Group']].rename(columns={'Feature_Top_2': 'Feature', 'Value_Top_2': 'SHAP_Value'}),
        df[['Feature_Top_3', 'Value_Top_3', 'Group']].rename(columns={'Feature_Top_3': 'Feature', 'Value_Top_3': 'SHAP_Value'})
    ])

    # Convert SHAP values to absolute values
    feature_shap_values['SHAP_Value'] = feature_shap_values['SHAP_Value'].abs()
    # Calculate mean SHAP values and sort by descending order
    feature_shap_values = feature_shap_values.groupby(['Feature', 'Group'])['SHAP_Value'].mean().reset_index()
    feature_shap_values = feature_shap_values.sort_values(by='SHAP_Value', ascending=False).reset_index(drop=True)

    group_feat = list(set(df['Group']))
    if group_feat[0] == 'Older' and len(group_feat) == 1:
        return feature_shap_values['Feature'].to_numpy()

    plt.figure(figsize=(12, 8))
    sns.barplot(data=feature_shap_values, y='Feature', x='SHAP_Value', hue = 'Group', orient='h', ci=None)
    plt.title(title)
    plt.tight_layout()  # Adjust the layout to fit everything
    plt.savefig(filename, bbox_inches='tight')
    plt.close()

path_folder = f'Sample Data'
#path_folder = f'Variant Speeds Young and Older'

# Create a directory to save the plots
output_dir = f'{path_folder}/SHAP_Top_3'
os.makedirs(output_dir, exist_ok=True)

# READ AS EXCEL FILE
df_SHAP = pd.ExcelFile(f'{path_folder}/SHAP_DATA_older.xlsx')
# IDENTIFY THE DIFFERENT SPREADSHEETS ON THE FILE
sheet_names = df_SHAP.sheet_names
# READ EVERY SPREADSHEET AND SAVE THEM AS ITEM ON A DICTIONARY
dfs = {sheet: pd.read_excel(df_SHAP, sheet_name=sheet) for sheet in sheet_names}

# Dictionary to hold the data from each sheet
sheets_data   = {}
SHAP_DATA     = {}
features_sort = {}

for model_sheet in dfs:
    columns   = dfs[model_sheet].columns
    features  = dfs[model_sheet]

    feature_top_name  = []
    feature_top_value = []
    sum_feat          = []

    for index, row in features.iterrows():
        type_class  = row.Group
        feat_values = row.iloc[4:]
        sum_feat.append(feat_values.sum())

        if type_class == 'Control':
            sorted_feat_values = feat_values.sort_values(ascending=True)
        elif type_class == 'Flatfoot':
            sorted_feat_values = feat_values.sort_values(ascending=False)

        feature_top_value.append(list(sorted_feat_values.head(3)))
        feature_top_name.append(list(sorted_feat_values.head(3).index))

    sum_feat          = np.array(sum_feat)
    feature_top_name  = np.array(feature_top_name)
    feature_top_value = np.array(feature_top_value)

    dfs[model_sheet]['SUM'] = sum_feat

    new_labels = ['Feature_Top_1', 'Value_Top_1', 'Feature_Top_2', 'Value_Top_2', 'Feature_Top_3', 'Value_Top_3']
    counter    = 0
    for i in range(0,len(new_labels),2):
        dfs[model_sheet][new_labels[i]]   = feature_top_name[:,  counter]
        dfs[model_sheet][new_labels[i+1]] = feature_top_value[:, counter]
        counter += 1

    class1_df   = dfs[model_sheet][dfs[model_sheet]['Group'] == 'Control']
    class2_df   = dfs[model_sheet][dfs[model_sheet]['Group'] == 'Flatfoot']
    combined_df = dfs[model_sheet]

    plot_shap_ranking(class1_df, f"Modified SHAP Ranking for Control Class Model {model_sheet}", os.path.join(output_dir, f"SHAP_Top_{model_sheet}_Control.png"))
    # Plot for Older class
    features_sort[model_sheet] = plot_shap_ranking(class2_df, f"Modified SHAP Ranking for Flatfoot Class Model {model_sheet}", os.path.join(output_dir, f"SHAP_Top_{model_sheet}_Flatfoot.png"))
    # Plot for Combined (absolute values)
    plot_shap_ranking(combined_df, f"Modified SHAP Ranking for All (Absolute Values) from Model {model_sheet}", os.path.join(output_dir, f"SHAP_Top_{model_sheet}_All.png"))


Features_sort = pd.DataFrame.from_dict(features_sort, orient='index').transpose()
Features_sort.to_excel(f'{path_folder}/Features_SHAP_TOP.xlsx', index=True)
# Save the updated DataFrames back to an Excel file with multiple sheets
with pd.ExcelWriter(f'{path_folder}/TOPs_SHAP_DATA.xlsx') as writer:
    for sheet_name, df in dfs.items():
        df.to_excel(writer, sheet_name=sheet_name, index=True)

<ipython-input-3-3840631030>:33: FutureWarning: 

The `ci` parameter is deprecated. Use `errorbar=None` for the same effect.

  sns.barplot(data=feature_shap_values, y='Feature', x='SHAP_Value', hue = 'Group', orient='h', ci=None)
<ipython-input-3-3840631030>:33: FutureWarning: 

The `ci` parameter is deprecated. Use `errorbar=None` for the same effect.

  sns.barplot(data=feature_shap_values, y='Feature', x='SHAP_Value', hue = 'Group', orient='h', ci=None)
<ipython-input-3-3840631030>:33: FutureWarning: 

The `ci` parameter is deprecated. Use `errorbar=None` for the same effect.

  sns.barplot(data=feature_shap_values, y='Feature', x='SHAP_Value', hue = 'Group', orient='h', ci=None)
<ipython-input-3-3840631030>:33: FutureWarning: 

The `ci` parameter is deprecated. Use `errorbar=None` for the same effect.

  sns.barplot(data=feature_shap_values, y='Feature', x='SHAP_Value', hue = 'Group', orient='h', ci=None)
<ipython-input-3-3840631030>:33: FutureWarning: 

The `ci` parameter is depr